# LeetCode 999 — Available Captures for Rook bằng Python

**Link đề:** https://leetcode.com/problems/available-captures-for-rook/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix, Simulation

```python
class Solution:
    def numRookCaptures(self, board: list[list[str]]) -> int:
        ...
```

## Mục tiêu học
1. Làm việc với **ma trận** (list lồng list): chỉ số `board[r][c]`, duyệt hàng/cột, kiểm tra biên.
2. Nắm kỹ thuật **vector hướng** `(dr, dc)` để đi theo 4 hướng bằng **một** đoạn code thay vì viết 4 lần.
3. Hiểu mô phỏng "đi đến khi gặp vật cản" bằng `while` + `break`.
4. Viết 4 cách: kiểm tra từng con tốt (baseline), đi 4 hướng thủ công, vector hướng, xử lý chuỗi.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Bàn cờ `8 × 8`: có đúng **một** xe trắng `'R'`, một số tượng trắng `'B'`, một số tốt đen `'p'`, ô trống `'.'`.

Xe đi **ngang hoặc dọc** bao nhiêu ô cũng được, cho đến khi gặp quân khác hoặc mép bàn. Xe **tấn công** một con tốt nếu đi được đến ô của con tốt trong **một nước**. Xe **không** đi xuyên qua quân khác (tượng hay tốt đều chặn đường).

Trả về số con tốt mà xe đang tấn công.

- Ví dụ 1: xe ở hàng 2, cột 3; tốt ở (1,3), (2,7), (5,3) → đường đều thông → **3**.
- Ví dụ 2: xe bị 4 quân tượng bao quanh → **0**.
- Ví dụ 3: xe ở (3,3). Phía trên có hai con tốt (2,3) và (1,3) — xe chỉ ăn được con **gần hơn** (2,3); bên trái ăn (3,1) (con (3,0) bị che); bên phải ăn (3,5); phía dưới bị tượng (5,3) chặn → **3**.

**Constraints:** `board` là `8 × 8`; mỗi ô thuộc `{'R', '.', 'B', 'p'}`; đúng một `'R'`.

(Ghi chú: hàng đầu tiên của Ví dụ 2 trên trang đề in thiếu một ô; dữ liệu test chính thức có đủ 8 ô — notebook dùng bản đủ.)

In [1]:
board1 = [[".",".",".",".",".",".",".","."],[".",".",".","p",".",".",".","."],[".",".",".","R",".",".",".","p"],[".",".",".",".",".",".",".","."],[".",".",".",".",".",".",".","."],[".",".",".","p",".",".",".","."],[".",".",".",".",".",".",".","."],[".",".",".",".",".",".",".","."]]
board2 = [[".",".",".",".",".",".",".","."],[".","p","p","p","p","p",".","."],[".","p","p","B","p","p",".","."],[".","p","B","R","B","p",".","."],[".","p","p","B","p","p",".","."],[".","p","p","p","p","p",".","."],[".",".",".",".",".",".",".","."],[".",".",".",".",".",".",".","."]]
board3 = [[".",".",".",".",".",".",".","."],[".",".",".","p",".",".",".","."],[".",".",".","p",".",".",".","."],["p","p",".","R",".","p","B","."],[".",".",".",".",".",".",".","."],[".",".",".","B",".",".",".","."],[".",".",".","p",".",".",".","."],[".",".",".",".",".",".",".","."]]

def show_board(board):
    print("   " + " ".join(str(c) for c in range(8)))
    for r, row in enumerate(board):
        print(f"{r}  " + " ".join(row))

show_board(board3)

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . p . . . .
2  . . . p . . . .
3  p p . R . p B .
4  . . . . . . . .
5  . . . B . . . .
6  . . . p . . . .
7  . . . . . . . .


## 2. Nền tảng Python: ma trận

### 2.1. Chỉ số hai chiều

`board` là **list các hàng**; mỗi hàng là list 8 ký tự. `board[r]` là hàng `r`, `board[r][c]` là ô hàng `r` cột `c`.

- Đi **lên**: `r − 1`; **xuống**: `r + 1`; **trái**: `c − 1`; **phải**: `c + 1`.
- Ô hợp lệ khi `0 <= r < 8` và `0 <= c < 8`. Python cho phép viết **so sánh dây chuyền** `0 <= r < 8`.

In [2]:
print("board3[3] =", board3[3])
print("board3[3][3] =", board3[3][3])
r = 8
print("0 <= 8 < 8 ?", 0 <= r < 8)
print("Cảnh báo: board3[-1][0] =", repr(board3[-1][0]), "-> chỉ số âm KHÔNG báo lỗi mà lấy hàng cuối!")

board3[3] = ['p', 'p', '.', 'R', '.', 'p', 'B', '.']
board3[3][3] = R
0 <= 8 < 8 ? False
Cảnh báo: board3[-1][0] = '.' -> chỉ số âm KHÔNG báo lỗi mà lấy hàng cuối!


Dòng cuối rất quan trọng: chỉ số `-1` không gây lỗi trong Python mà âm thầm lấy phần tử cuối. Khi mô phỏng đi ra khỏi bàn cờ, **phải tự kiểm tra biên**, đừng trông chờ `IndexError`.

### 2.2. Tìm vị trí xe — duyệt hai vòng lặp và `enumerate`

In [3]:
def find_rook(board):
    for r, row in enumerate(board):
        for c, cell in enumerate(row):
            if cell == "R":
                return r, c

print("Xe ở:", find_rook(board1), find_rook(board2), find_rook(board3))

Xe ở: (2, 3) (3, 3) (3, 3)


Cách gọn hơn: `next((r, c) for r in range(8) for c in range(8) if board[r][c] == "R")` — `next` lấy phần tử đầu tiên của generator.

### 2.3. Vector hướng

Một hướng được mô tả bằng cặp `(dr, dc)`: mỗi bước cộng `dr` vào hàng và `dc` vào cột.

| Hướng | (dr, dc) |
|---|---|
| lên | (−1, 0) |
| xuống | (1, 0) |
| trái | (0, −1) |
| phải | (0, 1) |

In [4]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
r, c = 3, 3
for dr, dc in DIRS:
    path = []
    nr, nc = r + dr, c + dc
    while 0 <= nr < 8 and 0 <= nc < 8:
        path.append((nr, nc))
        nr, nc = nr + dr, nc + dc
    print(f"hướng ({dr:2},{dc:2}): {path}")

hướng (-1, 0): [(2, 3), (1, 3), (0, 3)]
hướng ( 1, 0): [(4, 3), (5, 3), (6, 3), (7, 3)]
hướng ( 0,-1): [(3, 2), (3, 1), (3, 0)]
hướng ( 0, 1): [(3, 4), (3, 5), (3, 6), (3, 7)]


## 3. Cách trực giác (baseline) — xét từng con tốt

Với mỗi con tốt `p`:
- Nếu **cùng hàng** hoặc **cùng cột** với xe, kiểm tra mọi ô **nằm giữa** hai quân có trống không.
- Trống hết → xe ăn được con tốt đó.

In [5]:
def captures_brute(board):
    rr, rc = find_rook(board)
    count = 0
    for r in range(8):
        for c in range(8):
            if board[r][c] != "p":
                continue
            if r == rr:
                lo, hi = sorted((c, rc))
                between = [board[r][k] for k in range(lo + 1, hi)]
            elif c == rc:
                lo, hi = sorted((r, rr))
                between = [board[k][c] for k in range(lo + 1, hi)]
            else:
                continue                               # khác hàng & cột
            if all(x == "." for x in between):
                count += 1
    return count

print(captures_brute(board1), captures_brute(board2), captures_brute(board3))

3 0 3


Bàn cờ cố định 8×8 nên mọi cách đều O(1) theo nghĩa chặt. Tổng quát cho bàn `N × N` với `P` con tốt: baseline tốn O(N² + P·N).

## 4. Nút thắt

Baseline nhìn bài theo hướng "**con tốt nào** bị ăn" — phải xét mọi con tốt và kiểm tra đoạn ở giữa. Nhưng xe chỉ có thể ăn **tối đa 4** con (một con mỗi hướng), và là con **đầu tiên** gặp trên mỗi hướng.

## 5. Insight — đứng ở xe, nhìn ra 4 hướng

Từ ô của xe, đi theo từng hướng:
- Gặp `'.'` → đi tiếp.
- Gặp `'p'` → ăn được, **cộng 1 và dừng** hướng này.
- Gặp `'B'` → bị chặn, **dừng** hướng này.
- Ra khỏi bàn → dừng.

Chỉ quan tâm **quân đầu tiên** trên mỗi hướng. Kết quả nằm trong `0..4`.

## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1:** tìm `(rr, rc)` của xe.

**Đoạn 2:** với mỗi `(dr, dc)` trong `DIRS`, bắt đầu từ ô kế bên: `r, c = rr + dr, rc + dc`.

**Đoạn 3:** `while` còn trong bàn:
- `'B'` → `break`; `'p'` → `count += 1` rồi `break`; ngược lại đi tiếp.

In [6]:
def num_rook_captures(board: list[list[str]]) -> int:
    rr, rc = find_rook(board)                       # Đoạn 1
    count = 0
    for dr, dc in DIRS:                             # Đoạn 2
        r, c = rr + dr, rc + dc
        while 0 <= r < 8 and 0 <= c < 8:            # Đoạn 3
            if board[r][c] == "B":
                break
            if board[r][c] == "p":
                count += 1
                break
            r, c = r + dr, c + dc
    return count

print(num_rook_captures(board1), num_rook_captures(board2), num_rook_captures(board3))

3 0 3


## 7. Trace từng bước

`board3`, xe ở `(3, 3)`:

| Hướng | Ô lần lượt | Quân đầu tiên | Kết quả |
|---|---|---|---|
| lên (−1,0) | (2,3) | `p` | +1 → count = 1 |
| xuống (1,0) | (4,3) `.`, (5,3) | `B` | chặn |
| trái (0,−1) | (3,2) `.`, (3,1) | `p` | +1 → count = 2 (con tốt (3,0) bị che, không tính) |
| phải (0,1) | (3,4) `.`, (3,5) | `p` | +1 → count = 3 |

In [7]:
NAMES = {(-1, 0): "lên", (1, 0): "xuống", (0, -1): "trái", (0, 1): "phải"}

def trace_rook(board):
    rr, rc = find_rook(board)
    print("Xe ở", (rr, rc))
    count = 0
    for dr, dc in DIRS:
        r, c = rr + dr, rc + dc
        steps = []
        outcome = "ra khỏi bàn"
        while 0 <= r < 8 and 0 <= c < 8:
            steps.append(f"({r},{c}){board[r][c]}")
            if board[r][c] == "B":
                outcome = "bị tượng chặn"
                break
            if board[r][c] == "p":
                count += 1
                outcome = f"ĂN tốt -> count = {count}"
                break
            r, c = r + dr, c + dc
        print(f"  {NAMES[(dr, dc)]:5}: {' '.join(steps):40} => {outcome}")
    print("Tổng:", count)
    return count

trace_rook(board3)
print()
trace_rook(board2)

Xe ở (3, 3)
  lên  : (2,3)p                                   => ĂN tốt -> count = 1
  xuống: (4,3). (5,3)B                            => bị tượng chặn
  trái : (3,2). (3,1)p                            => ĂN tốt -> count = 2
  phải : (3,4). (3,5)p                            => ĂN tốt -> count = 3
Tổng: 3

Xe ở (3, 3)
  lên  : (2,3)B                                   => bị tượng chặn
  xuống: (4,3)B                                   => bị tượng chặn
  trái : (3,2)B                                   => bị tượng chặn
  phải : (3,4)B                                   => bị tượng chặn
Tổng: 0


0

## 8. Code nộp LeetCode

### Cách A — vector hướng

In [8]:
class Solution:
    def numRookCaptures(self, board: list[list[str]]) -> int:
        rr, rc = next((r, c) for r in range(8) for c in range(8) if board[r][c] == "R")
        count = 0
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            r, c = rr + dr, rc + dc
            while 0 <= r < 8 and 0 <= c < 8 and board[r][c] != "B":
                if board[r][c] == "p":
                    count += 1
                    break
                r, c = r + dr, c + dc
        return count

s = Solution()
print(s.numRookCaptures(board1), s.numRookCaptures(board2), s.numRookCaptures(board3))

3 0 3


Ở đây điều kiện "gặp tượng thì dừng" được gộp vào điều kiện `while` — gọn hơn nhưng vẫn cùng logic.

### Cách B — viết rõ 4 vòng lặp (dễ hiểu cho người mới)

In [9]:
class SolutionExplicit:
    def numRookCaptures(self, board: list[list[str]]) -> int:
        for i in range(8):
            for j in range(8):
                if board[i][j] == "R":
                    rr, rc = i, j
        count = 0
        for r in range(rr - 1, -1, -1):          # lên
            if board[r][rc] != ".":
                count += board[r][rc] == "p"
                break
        for r in range(rr + 1, 8):               # xuống
            if board[r][rc] != ".":
                count += board[r][rc] == "p"
                break
        for c in range(rc - 1, -1, -1):          # trái
            if board[rr][c] != ".":
                count += board[rr][c] == "p"
                break
        for c in range(rc + 1, 8):               # phải
            if board[rr][c] != ".":
                count += board[rr][c] == "p"
                break
        return count

print(SolutionExplicit().numRookCaptures(board3))
print("True + 0 =", True + 0, "| False + 0 =", False + 0)

3
True + 0 = 1 | False + 0 = 0


`count += (điều kiện)` hoạt động vì trong Python `True == 1` và `False == 0` khi làm phép cộng.

### Cách C — xử lý chuỗi (sáng tạo)

Ghép hàng của xe thành chuỗi, ví dụ `"pp.R.pB."`, rồi **bỏ hết dấu chấm**: `"ppRpB"`. Xe ăn tốt bên trái nếu ký tự ngay trước `R` là `p`; ăn bên phải nếu ký tự ngay sau `R` là `p`. Cột làm tương tự.

`str.replace(".", "")` xoá mọi dấu chấm; `"".join(row)` ghép list ký tự thành chuỗi.

In [10]:
row = "".join(board3[3])
squeezed = row.replace(".", "")
print(repr(row), "->", repr(squeezed))
col = "".join(board3[r][3] for r in range(8))
print(repr(col), "->", repr(col.replace(".", "")))

'pp.R.pB.' -> 'ppRpB'
'.ppR.Bp.' -> 'ppRBp'


In [11]:
class SolutionString:
    def numRookCaptures(self, board: list[list[str]]) -> int:
        rr, rc = next((r, c) for r in range(8) for c in range(8) if board[r][c] == "R")
        lines = ["".join(board[rr]), "".join(board[r][rc] for r in range(8))]
        count = 0
        for line in lines:
            s = line.replace(".", "")
            k = s.index("R")
            count += k > 0 and s[k - 1] == "p"
            count += k + 1 < len(s) and s[k + 1] == "p"
        return count

print(SolutionString().numRookCaptures(board1), SolutionString().numRookCaptures(board2),
      SolutionString().numRookCaptures(board3))

3 0 3


Vì sao đúng: bỏ dấu chấm không đổi **thứ tự** các quân; hai quân "liền kề sau khi bỏ chấm" nghĩa là giữa chúng **chỉ toàn ô trống** trên bàn cờ — đúng điều kiện xe đi tới được.

## 9. Vì sao đúng? (Cách A)

- Xe chỉ di chuyển theo 4 hướng; một nước đi theo hướng `d` đến ô `(rr + k·dr, rc + k·dc)` hợp lệ ⇔ mọi ô `1..k−1` trên hướng đó trống và ô `k` trong bàn.
- Vòng `while` duyệt `k = 1, 2, ...` và dừng ở quân **đầu tiên**. Nếu là `p` → con tốt đó bị tấn công; mọi con tốt xa hơn trên cùng hướng bị nó che → không tính. Nếu là `B` → mọi thứ phía sau bị che. Ra khỏi bàn → không có quân nào.
- Một con tốt bị tấn công thì phải nằm trên một trong 4 hướng và là quân đầu tiên trên hướng đó → được đếm đúng một lần.

## 10. Độ phức tạp

Tổng quát bàn `N × N` (đề: `N = 8`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Baseline từng con tốt | O(N² + P·N) | O(N) (list `between`) | nhìn từ con tốt |
| **A. Vector hướng** | O(N²) tìm xe + O(N) đi 4 hướng | O(1) | gọn, mở rộng dễ (tượng, hậu) |
| B. 4 vòng tường minh | như A | O(1) | dài nhưng rõ |
| C. Chuỗi | O(N²) tìm xe + O(N) | O(N) | sáng tạo, ít lỗi biên |

Với `N = 8`, tất cả là hằng số — điểm quan trọng là **tính đúng** và **code sạch**.

## 11. Bộ test

In [12]:
def make_board(rows: list[str]) -> list[list[str]]:
    # Tiện dựng bàn cờ từ 8 chuỗi dài 8
    return [list(r) for r in rows]

edge_rook_corner = make_board(["R..p....", "........", "........", "p.......", "........", "........", "........", "........"])
edge_all_blocked = make_board(["........", "...B....", "..pBp...", "...R....", "...p....", "........", "........", "........"])
edge_empty = make_board(["........", "........", "........", "...R....", "........", "........", "........", "........"])
edge_four = make_board(["...p....", "........", "........", "p..R...p", "........", "........", "........", "...p...."])
edge_shadow = make_board(["...p....", "...p....", "........", "B.pR....", "........", "........", "........", "........"])

tests = [
    (board1, 3), (board2, 0), (board3, 3),
    (edge_rook_corner, 2),    # xe ở góc, tốt cùng hàng và cùng cột
    (edge_all_blocked, 1),    # lên bị B; trái/phải là ô trống ra mép; xuống ăn p
    (edge_empty, 0),
    (edge_four, 4),           # tối đa 4
    (edge_shadow, 2),         # tốt che tốt: chỉ đếm con gần nhất
]
impls = {
    "brute": captures_brute, "chính": num_rook_captures,
    "LC-A": Solution().numRookCaptures, "LC-B": SolutionExplicit().numRookCaptures,
    "LC-C": SolutionString().numRookCaptures,
}
for i, (board, expected) in enumerate(tests):
    for name, f in impls.items():
        got = f(board)
        assert got == expected, (name, i, got, expected)
    print(f"test {i}: rook {find_rook(board)} -> {expected} PASS")
print("Tất cả test PASS!")

test 0: rook (2, 3) -> 3 PASS
test 1: rook (3, 3) -> 0 PASS
test 2: rook (3, 3) -> 3 PASS
test 3: rook (0, 0) -> 2 PASS
test 4: rook (3, 3) -> 1 PASS
test 5: rook (3, 3) -> 0 PASS
test 6: rook (3, 3) -> 4 PASS
test 7: rook (3, 3) -> 2 PASS
Tất cả test PASS!


In [13]:
import random
rng = random.Random(999)
for _ in range(3000):
    cells = rng.choices([".", ".", ".", "B", "p"], k=64)
    cells[rng.randrange(64)] = "R"
    if cells.count("R") != 1:
        continue
    board = [cells[i * 8:(i + 1) * 8] for i in range(8)]
    expected = captures_brute(board)
    for name, f in impls.items():
        assert f(board) == expected, (name, board)
print("3000 bàn cờ ngẫu nhiên: mọi cách khớp")

3000 bàn cờ ngẫu nhiên: mọi cách khớp


`rng.choices(population, k=64)` lấy **có lặp lại** 64 phần tử (ô trống xuất hiện 3 lần trong population nên có xác suất cao hơn). `cells[i*8:(i+1)*8]` cắt list 64 ô thành từng hàng 8 ô.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Không dừng sau khi ăn tốt (đếm cả con tốt bị che)

In [14]:
def wrong_no_break_after_pawn(board):
    rr, rc = find_rook(board)
    count = 0
    for dr, dc in DIRS:
        r, c = rr + dr, rc + dc
        while 0 <= r < 8 and 0 <= c < 8:
            if board[r][c] == "B":
                break
            if board[r][c] == "p":
                count += 1          # quên break!
            r, c = r + dr, c + dc
    return count

print("Sai :", wrong_no_break_after_pawn(board3))
print("Đúng:", num_rook_captures(board3))

Sai : 5
Đúng: 3


**Vì sao sai:** ở `board3`, hướng lên có hai con tốt (2,3) và (1,3); hướng trái có (3,1) và (3,0). Con phía sau bị con phía trước che. **Sửa:** `break` ngay sau khi ăn.

### Lỗi 2: Quên kiểm tra biên, dựa vào chỉ số âm

In [15]:
def wrong_no_bounds(board):
    rr, rc = find_rook(board)
    count = 0
    for dr, dc in DIRS:
        r, c = rr + dr, rc + dc
        for _ in range(8):                  # đi đúng 8 bước, không kiểm tra biên
            if not (-8 <= r < 8 and -8 <= c < 8):
                break
            if board[r][c] == "B":
                break
            if board[r][c] == "p":
                count += 1
                break
            r, c = r + dr, c + dc
    return count

b_wrap = make_board(["........", "........", "........", "R......p", "........", "........", "........", "........"])
print("Sai :", wrong_no_bounds(b_wrap), "(đi sang trái từ cột 0 'quấn' sang cột 7 nhờ chỉ số âm)")
print("Đúng:", num_rook_captures(b_wrap))

Sai : 2 (đi sang trái từ cột 0 'quấn' sang cột 7 nhờ chỉ số âm)
Đúng: 1


**Vì sao sai:** ở bàn này tốt (3,7) cùng hàng và đường thông nên xe ăn được bằng hướng phải (đúng 1 lần). Nhưng bản sai khi đi sang **trái** từ cột 0 lại đọc `board[3][-1]`, tức là cột 7 → cùng con tốt bị đếm **2 lần**. **Sửa:** điều kiện `0 <= r < 8 and 0 <= c < 8`.

### Lỗi 3: Coi tượng là "trong suốt"

Chỉ dừng khi gặp tốt mà bỏ qua `'B'` → xe "đi xuyên" tượng. Đề nói rõ: mọi quân đều chặn.

### Lỗi 4: Nhầm `board[r][c]` với `board[c][r]`

`board[r]` là **hàng**. Đi dọc là thay đổi `r`, giữ nguyên `c`. Viết ngược sẽ đi sai hướng và cho kết quả sai trên bàn không đối xứng.

## 13. Mẹo nhận dạng pattern

- Bài **ma trận + di chuyển theo hướng** (xe, tượng, hậu, robot, tia sáng) → khai báo `DIRS` rồi dùng **một** vòng lặp cho mọi hướng.
- "Đi đến khi gặp vật cản" → `while trong_biên: if vật_cản: break; bước tiếp`.
- Chỉ cần "**quân đầu tiên**" theo mỗi hướng → `break` ngay khi gặp.
- Luôn tự viết hàm/điều kiện `in_bounds(r, c)`; đừng dựa vào `IndexError` vì chỉ số âm không báo lỗi.

## 14. Biến thể

1. **Tượng** đi chéo → `DIRS = [(-1,-1), (-1,1), (1,-1), (1,1)]`; **Hậu** → 8 hướng.
2. **LeetCode 1222 — Queens That Can Attack the King:** 8 hướng từ vua, tìm hậu đầu tiên mỗi hướng.
3. **LeetCode 3001 — Minimum Moves to Capture The Queen.**
4. **LeetCode 289 — Game of Life**, **LeetCode 200 — Number of Islands:** cùng kỹ thuật vector hướng cho ô lân cận.

In [16]:
def queen_targets(board, piece="Q"):
    # Biến thể: hậu (8 hướng) tấn công được những con tốt nào?
    qr, qc = next((r, c) for r in range(8) for c in range(8) if board[r][c] == piece)
    hits = []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == dc == 0:
                continue
            r, c = qr + dr, qc + dc
            while 0 <= r < 8 and 0 <= c < 8 and board[r][c] == ".":
                r, c = r + dr, c + dc
            if 0 <= r < 8 and 0 <= c < 8 and board[r][c] == "p":
                hits.append((r, c))
    return sorted(hits)

bq = make_board(["p.......", "........", "..p.....", "...Q...p", "........", ".....p..", "........", "...B...."])
print("Hậu ăn được:", queen_targets(bq))

Hậu ăn được: [(2, 2), (3, 7), (5, 5)]


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Đếm quân
Trả về dict đếm số `'p'`, `'B'`, `'.'` trên bàn.

In [17]:
def count_pieces(board: list[list[str]]) -> dict:
    # TODO
    pass

# assert count_pieces(board1) == {'.': 60, 'p': 3, 'R': 1}

### Bài 2 — Ô trong biên
Viết `in_bounds(r, c, n=8)`.

In [18]:
def in_bounds(r: int, c: int, n: int = 8) -> bool:
    # TODO
    pass

# assert in_bounds(0, 7) == True
# assert in_bounds(-1, 3) == False
# assert in_bounds(3, 8) == False

### Bài 3 — Số ô xe đi tới được
Trả về số ô **trống** mà xe có thể đi tới trong một nước (không tính ô có tốt).

In [19]:
def rook_moves(board: list[list[str]]) -> int:
    # TODO
    pass

# assert rook_moves(board3) == 3
# assert rook_moves(board2) == 0

### Bài 4 — Tượng ăn tốt
Giả sử thay `'R'` bằng quân tượng trắng `'W'` đi chéo; đếm số tốt bị tấn công.

In [20]:
def bishop_captures(board: list[list[str]]) -> int:
    # TODO
    pass

# bw = make_board(["p.......", "........", "..p.....", "...W....", "....B...", ".....p..", "p.......", "........"])
# assert bishop_captures(bw) == 2

### Bài 5 — Nhiều xe
Bàn có thể có **nhiều** xe `'R'`. Trả về số con tốt **khác nhau** bị ít nhất một xe tấn công (xe cũng chặn đường xe khác).

In [21]:
def multi_rook_captures(board: list[list[str]]) -> int:
    # TODO
    pass

# bm = make_board(["R..p....", "........", "........", "p.......", "...R....", "........", "........", "........"])
# assert multi_rook_captures(bm) == 2

### Bài 6 (nâng cao) — Queens That Can Attack the King
Cho list toạ độ các hậu `queens` và vua `king` trên bàn 8×8, trả về các hậu tấn công được vua (sắp xếp tăng).

In [22]:
def queens_attack_king(queens: list[list[int]], king: list[int]) -> list[list[int]]:
    # TODO
    pass

# assert queens_attack_king([[0,1],[1,0],[4,0],[0,4],[3,3],[2,4]], [0,0]) == [[0, 1], [1, 0], [3, 3]]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | duyệt mọi ô | `Counter` | `Counter(cell for row in board for cell in row)` |
| 2 | so sánh dây chuyền | | `0 <= r < n and 0 <= c < n` |
| 3 | như lời giải chính | đếm mỗi ô `'.'` đi qua | dừng ở bất kỳ quân nào |
| 4 | đổi `DIRS` | 4 hướng chéo | phần còn lại giống hệt |
| 5 | lặp qua mọi xe | dùng `set` toạ độ tốt bị ăn | `R` khác cũng là vật cản (khác `'.'`) |
| 6 | đứng ở **vua** nhìn 8 hướng | set toạ độ hậu | hậu đầu tiên mỗi hướng |

## 17. Đáp án tham khảo

In [23]:
from collections import Counter

def ans_count_pieces(board):
    return dict(Counter(cell for row in board for cell in row))

def ans_in_bounds(r, c, n=8):
    return 0 <= r < n and 0 <= c < n

def ans_rook_moves(board):
    rr, rc = find_rook(board)
    total = 0
    for dr, dc in DIRS:
        r, c = rr + dr, rc + dc
        while ans_in_bounds(r, c) and board[r][c] == ".":
            total += 1
            r, c = r + dr, c + dc
    return total

def ans_bishop_captures(board):
    br, bc = next((r, c) for r in range(8) for c in range(8) if board[r][c] == "W")
    count = 0
    for dr, dc in ((-1, -1), (-1, 1), (1, -1), (1, 1)):
        r, c = br + dr, bc + dc
        while ans_in_bounds(r, c) and board[r][c] == ".":
            r, c = r + dr, c + dc
        if ans_in_bounds(r, c) and board[r][c] == "p":
            count += 1
    return count

def ans_multi_rook_captures(board):
    hit = set()
    for rr in range(8):
        for rc in range(8):
            if board[rr][rc] != "R":
                continue
            for dr, dc in DIRS:
                r, c = rr + dr, rc + dc
                while ans_in_bounds(r, c) and board[r][c] == ".":
                    r, c = r + dr, c + dc
                if ans_in_bounds(r, c) and board[r][c] == "p":
                    hit.add((r, c))
    return len(hit)

def ans_queens_attack_king(queens, king):
    qs = {tuple(q) for q in queens}
    res = []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == dc == 0:
                continue
            r, c = king[0] + dr, king[1] + dc
            while ans_in_bounds(r, c):
                if (r, c) in qs:
                    res.append([r, c])
                    break
                r, c = r + dr, c + dc
    return sorted(res)

assert ans_count_pieces(board1) == {'.': 60, 'p': 3, 'R': 1}
assert ans_in_bounds(0, 7) == True
assert ans_in_bounds(-1, 3) == False
assert ans_in_bounds(3, 8) == False
assert ans_rook_moves(board3) == 3
assert ans_rook_moves(board2) == 0
bw = make_board(["p.......", "........", "..p.....", "...W....", "....B...", ".....p..", "p.......", "........"])
assert ans_bishop_captures(bw) == 2
bm = make_board(["R..p....", "........", "........", "p.......", "...R....", "........", "........", "........"])
assert ans_multi_rook_captures(bm) == 2
assert ans_queens_attack_king([[0,1],[1,0],[4,0],[0,4],[3,3],[2,4]], [0,0]) == [[0, 1], [1, 0], [3, 3]]
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numRookCaptures(self, board: list[list[str]]) -> int:
        rr, rc = next((r, c) for r in range(8) for c in range(8) if board[r][c] == "R")
        count = 0
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            r, c = rr + dr, rc + dc
            while 0 <= r < 8 and 0 <= c < 8 and board[r][c] == ".":
                r, c = r + dr, c + dc
            if 0 <= r < 8 and 0 <= c < 8 and board[r][c] == "p":
                count += 1
        return count
```

Khuôn "quân đầu tiên trên mỗi hướng": **trượt qua ô trống → nhìn ô dừng lại**.

**Checklist trước khi submit**
- [ ] Kiểm tra biên `0 <= r < 8 and 0 <= c < 8` (không dựa vào chỉ số âm)?
- [ ] Dừng ở quân **đầu tiên** (tốt hoặc tượng)?
- [ ] Bắt đầu từ ô **kế bên** xe, không phải ô của xe?
- [ ] `board[r][c]`: `r` là hàng, `c` là cột?
- [ ] Kết quả trong `0..4`?